In [6]:
# Connect to the sqlitesearch database (faq.db)

from sqlitesearch import TextSearchIndex

# define sqlite_index
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

# check how many docs are in the index
# this will only load after 'persistent_rag_ingest.ipynb' finishes
sqlite_index.count()

79

In [7]:
# Try a search:
results = sqlite_index.search(
    "Can I still join the course after it started?",
    num_results=5
    )

# List of the "question" text in each of the docs that makes it to results 
[doc["question"] for doc in results]


['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I am using Azure OpenAI and I am still getting an error of Error code: 400 - {\'error\': {\'message\': "Missing required parameter: \'tools[0].function\'.", \'type\': \'invalid_request_error\', \'param\': \'tools[0].function\', \'code\': \'missing_required_parameter\'}}?']

In [8]:
# Use RAGBase from 'rag_helper.py' with sqlitesearch
# because RAG is modular, just swap the search index. Keep the rest the same.

# Import RAGBase from rag_helper.py & Anthropic LLM from anthropic
from rag_helper import RAGBase
from anthropic import Anthropic

# Initialize Anthropic client (picks API key from .env)
anthropic_client = Anthropic()

# Create the assistant using RAGBase class
assistant = RAGBase(
    index=sqlite_index,
    llm_client=anthropic_client,
)

In [9]:
# Test the new RAG assistant

answer = assistant.rag("Can I still join the course after it started?")
print(answer)

# Can I still join the course after it started?

Yes, you can still join the course after it has started. However, there's an important condition:

**If you want to receive a certificate**, you need to submit your capstone project while submissions are still being accepted.

Keep in mind that to earn a certificate, you must complete the course with a "live" cohort (not in self-paced mode), as you'll need to peer-review 3 capstone projects after submitting your own. Peer-reviewing can only be done while the course is actively running.


In [10]:
# Close the database connection
sqlite_index.close()